In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pod_models
from matplotlib.lines import Line2D

plt.rcParams.update({
    'axes.titlesize': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi': 600,
    'figure.constrained_layout.use': True,
})

color_Campbell = "#b6321c" # BrickRed
color_DNVGL = "#f58137" # Orange
color_CV = "#000000"
color_prior = "#0071bc" # RoyalBlue

color_0_5 = "#C0C0C0" # light gray
color_1_5 = "#808080" # medium gray
color_3_0 = "#404040" # dark gray
color_17_0 = "#000000" # black

color_hit  = "#006d02" # green
color_miss = "#a60014" # red
color_fp = "#0043a0" # blue

text_width = 5.3 # Document text width in inches

In [ ]:
all_data = pd.read_excel("pod_data.xlsx")
N = len(all_data)

A_unsorted = all_data["length_mm"].to_numpy()
order = np.argsort(A_unsorted)
A = A_unsorted[order]
R = (all_data["length_pix"] / all_data["length_mm"])[order]
A_hat = all_data["Detection_hit_miss"].to_numpy()[order]
decision_boundary = 0.1
Z = A_hat >= decision_boundary

X_A = np.log(A)[..., None]
θ_0_MLE_A, Θ_MLE_A = pod_models.fit_model(X_A, Z)
print(f"beta_0 = {θ_0_MLE_A:.3f}, beta_1 = {Θ_MLE_A[0]:.3f}")

# Binning Model

In [ ]:
bin_0_5 = np.logical_and(0 <= R, R < 1)
bin_1_5 = np.logical_and(1 <= R, R < 2)
bin_3_0 = np.logical_and(2 <= R, R < 4)
bin_17_0 = np.logical_and(4 <= R, R < 30)

X_A_0_5 = X_A[bin_0_5]
Z_0_5 = Z[bin_0_5]
θ_0_MLE_A_0_5, Θ_MLE_A_0_5 = pod_models.fit_model(X_A_0_5, Z_0_5)
print(f"bin 0.5: beta_0 = {θ_0_MLE_A_0_5:.3f}, beta_1 = {Θ_MLE_A_0_5[0]:.3f}")
X_A_1_5 = X_A[bin_1_5]
Z_1_5 = Z[bin_1_5]
θ_0_MLE_A_1_5, Θ_MLE_A_1_5 = pod_models.fit_model(X_A_1_5, Z_1_5)
print(f"bin 1.5: beta_0 = {θ_0_MLE_A_1_5:.3f}, beta_1 = {Θ_MLE_A_1_5[0]:.3f}")
X_A_3_0 = X_A[bin_3_0]
Z_3_0 = Z[bin_3_0]
θ_0_MLE_A_3_0, Θ_MLE_A_3_0 = pod_models.fit_model(X_A_3_0, Z_3_0)
print(f"bin 3.0: beta_0 = {θ_0_MLE_A_3_0:.3f}, beta_1 = {Θ_MLE_A_3_0[0]:.3f}")
X_A_17_0 = X_A[bin_17_0]
Z_17_0 = Z[bin_17_0]
θ_0_MLE_A_17_0, Θ_MLE_A_17_0 = pod_models.fit_model(X_A_17_0, Z_17_0)
print(f"bin 17.0: beta_0 = {θ_0_MLE_A_17_0:.3f}, beta_1 = {Θ_MLE_A_17_0[0]:.3f}")

In [ ]:
grid_samples = 80
θ_0_min, θ_0_max = θ_0_MLE_A_0_5 - 4, θ_0_MLE_A_0_5 + 4
Θ_min, Θ_max = [θ - 0.75 for θ in Θ_MLE_A_0_5], [θ + 0.75 for θ in Θ_MLE_A_0_5]
θ_0s = np.linspace(θ_0_min, θ_0_max, grid_samples)
Θs = [np.linspace(θ_min, θ_max, grid_samples) for (θ_min, θ_max) in zip(Θ_min, Θ_max)]
θ_0_mesh_0_5, *Θ_mesh_0_5 = np.meshgrid(θ_0s, *Θs, indexing="ij")
Λs_0_5 = pod_models.negative_log_likelihood_ratio(Z_0_5, X_A_0_5, θ_0_mesh_0_5[..., None], np.stack(Θ_mesh_0_5, axis=-1)[..., None, :], θ_0_MLE_A_0_5, Θ_MLE_A_0_5)
confidence_region_0_5 = Λs_0_5 <= pod_models.χ_sq(2, 0.05)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 5), layout="constrained")
cbar = ax[0].imshow(np.clip(Λs_0_5.T, 0., 40.), origin="lower", extent=[θ_0_min, θ_0_max, Θ_min[0], Θ_max[0]], aspect="auto")
contours = ax[0].contour(np.clip(Λs_0_5.T, 0., 40.), origin="lower", extent=[θ_0_min, θ_0_max, Θ_min[0], Θ_max[0]], levels=[pod_models.χ_sq(2, 0.05)], cmap="cool")
ax[0].clabel(contours, contours.levels, fmt={contours.levels[0]: r"$\chi_{2,0.95}^2$"})
ax[0].set_xlabel(r"$\theta_0$")
ax[0].set_ylabel(r"$\theta_1$")
ax[0].set_title(r"$\Lambda$")
fig.colorbar(cbar, ax=ax[0])
ax[1].imshow(confidence_region_0_5.T, origin="lower", extent=[θ_0_min, θ_0_max, Θ_min[0], Θ_max[0]], aspect="auto", cmap="gray")
ax[1].set_xlabel(r"$\theta_0$")
ax[1].set_ylabel(r"$\theta_1$")
ax[1].set_title(r"$\Lambda \leq \chi_{2,0.95}^2$");

In [ ]:
grid_samples = 80
θ_0_min, θ_0_max = θ_0_MLE_A_1_5 - 10, θ_0_MLE_A_1_5 + 6
Θ_min, Θ_max = [θ - 1.5 for θ in Θ_MLE_A_1_5], [θ + 2.25 for θ in Θ_MLE_A_1_5]
θ_0s = np.linspace(θ_0_min, θ_0_max, grid_samples)
Θs = [np.linspace(θ_min, θ_max, grid_samples) for (θ_min, θ_max) in zip(Θ_min, Θ_max)]
θ_0_mesh_1_5, *Θ_mesh_1_5 = np.meshgrid(θ_0s, *Θs, indexing="ij")
Λs_1_5 = pod_models.negative_log_likelihood_ratio(Z_1_5, X_A_1_5, θ_0_mesh_1_5[..., None], np.stack(Θ_mesh_1_5, axis=-1)[..., None, :], θ_0_MLE_A_1_5, Θ_MLE_A_1_5)
confidence_region_1_5 = Λs_1_5 <= pod_models.χ_sq(2, 0.05)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 5), layout="constrained")
cbar = ax[0].imshow(np.clip(Λs_1_5.T, 0., 40.), origin="lower", extent=[θ_0_min, θ_0_max, Θ_min[0], Θ_max[0]], aspect="auto")
contours = ax[0].contour(np.clip(Λs_1_5.T, 0., 40.), origin="lower", extent=[θ_0_min, θ_0_max, Θ_min[0], Θ_max[0]], levels=[pod_models.χ_sq(2, 0.05)], cmap="cool")
ax[0].clabel(contours, contours.levels, fmt={contours.levels[0]: r"$\chi_{2,0.95}^2$"})
ax[0].set_xlabel(r"$\theta_0$")
ax[0].set_ylabel(r"$\theta_1$")
ax[0].set_title(r"$\Lambda$")
fig.colorbar(cbar, ax=ax[0])
ax[1].imshow(confidence_region_1_5.T, origin="lower", extent=[θ_0_min, θ_0_max, Θ_min[0], Θ_max[0]], aspect="auto", cmap="gray")
ax[1].set_xlabel(r"$\theta_0$")
ax[1].set_ylabel(r"$\theta_1$")
ax[1].set_title(r"$\Lambda \leq \chi_{2,0.95}^2$");

In [ ]:
grid_samples = 80
θ_0_min, θ_0_max = θ_0_MLE_A_3_0 - 13, θ_0_MLE_A_3_0 + 8
Θ_min, Θ_max = [θ - 1.5 for θ in Θ_MLE_A_3_0], [θ + 3 for θ in Θ_MLE_A_3_0]
θ_0s = np.linspace(θ_0_min, θ_0_max, grid_samples)
Θs = [np.linspace(θ_min, θ_max, grid_samples) for (θ_min, θ_max) in zip(Θ_min, Θ_max)]
θ_0_mesh_3_0, *Θ_mesh_3_0 = np.meshgrid(θ_0s, *Θs, indexing="ij")
Λs_3_0 = pod_models.negative_log_likelihood_ratio(Z_3_0, X_A_3_0, θ_0_mesh_3_0[..., None], np.stack(Θ_mesh_3_0, axis=-1)[..., None, :], θ_0_MLE_A_3_0, Θ_MLE_A_3_0)
confidence_region_3_0 = Λs_3_0 <= pod_models.χ_sq(2, 0.05)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 5), layout="constrained")
cbar = ax[0].imshow(np.clip(Λs_3_0.T, 0., 40.), origin="lower", extent=[θ_0_min, θ_0_max, Θ_min[0], Θ_max[0]], aspect="auto")
contours = ax[0].contour(np.clip(Λs_3_0.T, 0., 40.), origin="lower", extent=[θ_0_min, θ_0_max, Θ_min[0], Θ_max[0]], levels=[pod_models.χ_sq(2, 0.05)], cmap="cool")
ax[0].clabel(contours, contours.levels, fmt={contours.levels[0]: r"$\chi_{2,0.95}^2$"})
ax[0].set_xlabel(r"$\theta_0$")
ax[0].set_ylabel(r"$\theta_1$")
ax[0].set_title(r"$\Lambda$")
fig.colorbar(cbar, ax=ax[0])
ax[1].imshow(confidence_region_3_0.T, origin="lower", extent=[θ_0_min, θ_0_max, Θ_min[0], Θ_max[0]], aspect="auto", cmap="gray")
ax[1].set_xlabel(r"$\theta_0$")
ax[1].set_ylabel(r"$\theta_1$")
ax[1].set_title(r"$\Lambda \leq \chi_{2,0.95}^2$");

In [ ]:
grid_samples = 80
θ_0_min, θ_0_max = θ_0_MLE_A_17_0 - 5, θ_0_MLE_A_17_0 + 6
Θ_min, Θ_max = [θ - 1 for θ in Θ_MLE_A_17_0], [θ + 1.25 for θ in Θ_MLE_A_17_0]
θ_0s = np.linspace(θ_0_min, θ_0_max, grid_samples)
Θs = [np.linspace(θ_min, θ_max, grid_samples) for (θ_min, θ_max) in zip(Θ_min, Θ_max)]
θ_0_mesh_17_0, *Θ_mesh_17_0 = np.meshgrid(θ_0s, *Θs, indexing="ij")
Λs_17_0 = pod_models.negative_log_likelihood_ratio(Z_17_0, X_A_17_0, θ_0_mesh_17_0[..., None], np.stack(Θ_mesh_17_0, axis=-1)[..., None, :], θ_0_MLE_A_17_0, Θ_MLE_A_17_0)
confidence_region_17_0 = Λs_17_0 <= pod_models.χ_sq(2, 0.05)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 5), layout="constrained")
cbar = ax[0].imshow(np.clip(Λs_17_0.T, 0., 40.), origin="lower", extent=[θ_0_min, θ_0_max, Θ_min[0], Θ_max[0]], aspect="auto")
contours = ax[0].contour(np.clip(Λs_17_0.T, 0., 40.), origin="lower", extent=[θ_0_min, θ_0_max, Θ_min[0], Θ_max[0]], levels=[pod_models.χ_sq(2, 0.05)], cmap="cool")
ax[0].clabel(contours, contours.levels, fmt={contours.levels[0]: r"$\chi_{2,0.95}^2$"})
ax[0].set_xlabel(r"$\theta_0$")
ax[0].set_ylabel(r"$\theta_1$")
ax[0].set_title(r"$\Lambda$")
fig.colorbar(cbar, ax=ax[0])
ax[1].imshow(confidence_region_17_0.T, origin="lower", extent=[θ_0_min, θ_0_max, Θ_min[0], Θ_max[0]], aspect="auto", cmap="gray")
ax[1].set_xlabel(r"$\theta_0$")
ax[1].set_ylabel(r"$\theta_1$")
ax[1].set_title(r"$\Lambda \leq \chi_{2,0.95}^2$");

In [ ]:
# We don't plot the confidence bounds here because the figure would become too messy.
θ_0_confidence_0_5, Θ_confidence_0_5 = θ_0_mesh_0_5[confidence_region_0_5], [θ[confidence_region_0_5] for θ in Θ_mesh_0_5]
θ_0_confidence_1_5, Θ_confidence_1_5 = θ_0_mesh_1_5[confidence_region_1_5], [θ[confidence_region_1_5] for θ in Θ_mesh_1_5]
θ_0_confidence_3_0, Θ_confidence_3_0 = θ_0_mesh_3_0[confidence_region_3_0], [θ[confidence_region_3_0] for θ in Θ_mesh_3_0]
θ_0_confidence_17_0, Θ_confidence_17_0 = θ_0_mesh_17_0[confidence_region_17_0], [θ[confidence_region_17_0] for θ in Θ_mesh_17_0]

## Figure 13

In [ ]:
As = np.linspace(1, 1500, 500)

fig, ax = plt.subplots(1, 1, figsize=(text_width, 0.6 * text_width))
ax.grid(True, alpha=0.3)

ax.plot(As, pod_models.PoD(np.log(As)[..., None], θ_0_MLE_A_17_0, Θ_MLE_A_17_0), color=color_17_0, label="$r = 17.0$ pix/mm")
ax.plot(As, pod_models.PoD(np.log(As)[..., None], θ_0_MLE_A_3_0, Θ_MLE_A_3_0), color=color_3_0, label="$r = 3.0$ pix/mm")
ax.plot(As, pod_models.PoD(np.log(As)[..., None], θ_0_MLE_A_1_5, Θ_MLE_A_1_5), color=color_1_5, label="$r = 1.5$ pix/mm")
ax.plot(As, pod_models.PoD(np.log(As)[..., None], θ_0_MLE_A_0_5, Θ_MLE_A_0_5), color=color_0_5, label="$r = 0.5$ pix/mm")
ax.plot(As, pod_models.DNVGL(As), color=color_DNVGL, linestyle=(5, (2, 3)), label="DNVGL")
ax.plot(As, pod_models.Campbell(As), color=color_Campbell, linestyle=(5, (2, 3)), label="Campbell")

ax.set_xticks(np.arange(0, 1501, 100))
ax.set_yticks(np.linspace(0, 1, 11))
ax.set_xlim([-50, 1500])
ax.tick_params(axis='x', rotation=45)
ax.set_xlabel("Crack Length (mm)")
ax.set_ylabel("Probability of Detection")
ax.legend(loc='lower right')
fig.savefig("Figures/PoD_resolution_binning.png")

# Parametric Model

In [ ]:
X_AR = np.stack([np.log(A), np.log(R)], axis=-1)
θ_0_MLE_AR, Θ_MLE_AR = pod_models.fit_model(X_AR, Z)
print(f"beta_0 = {θ_0_MLE_AR:.3f}, beta_1 = {Θ_MLE_AR[0]:.3f}, beta_2 = {Θ_MLE_AR[1]:.3f}")

ll_A = pod_models.log_likelihood(Z, X_A, θ_0_MLE_A, Θ_MLE_A)
ll_AR = pod_models.log_likelihood(Z, X_AR, θ_0_MLE_AR, Θ_MLE_AR)
Λ = -2 * (ll_A - ll_AR)
print(f"Likelihood ratio statistic: {Λ:.2f}. Reject null hypothesis: {Λ > pod_models.χ_sq(1, 0.05)}.")
print(f"p-value: {pod_models.χ_sq_p(1, Λ):.2f}")

In [ ]:
grid_samples = 80
θ_0_min, θ_0_max = θ_0_MLE_AR - 2, θ_0_MLE_AR + 3
Θ_min, Θ_max = [θ - 0.6 for θ in Θ_MLE_AR], [θ + 0.5 for θ in Θ_MLE_AR]
θ_0s = np.linspace(θ_0_min, θ_0_max, grid_samples)
Θs = [np.linspace(θ_min, θ_max, grid_samples) for (θ_min, θ_max) in zip(Θ_min, Θ_max)]
θ_0_mesh, *Θ_mesh = np.meshgrid(θ_0s, *Θs, indexing="ij")
Λs = pod_models.negative_log_likelihood_ratio(Z, X_AR, θ_0_mesh[..., None], np.stack(Θ_mesh, axis=-1)[..., None, :], θ_0_MLE_AR, Θ_MLE_AR)
confidence_region = Λs <= pod_models.χ_sq(3, 0.05)

In [ ]:
k = 20
Λs_slice = Λs[..., k]
confidence_region_slice = confidence_region[..., k]
fig, ax = plt.subplots(1, 2, figsize=(11, 5), layout="constrained")
cbar = ax[0].imshow(np.clip(Λs_slice.T, 0., 40.), origin="lower", extent=[θ_0_min, θ_0_max, Θ_min[0], Θ_max[0]], aspect="auto")
contours = ax[0].contour(np.clip(Λs_slice.T, 0., 40.), origin="lower", extent=[θ_0_min, θ_0_max, Θ_min[0], Θ_max[0]], levels=[pod_models.χ_sq(3, 0.05)], cmap="cool")
ax[0].clabel(contours, contours.levels, fmt={contours.levels[0]: r"$\chi_{3,0.95}^2$"})
ax[0].set_xlabel(r"$\theta_0$")
ax[0].set_ylabel(r"$\theta_1$")
ax[0].set_title(r"$\Lambda$")
fig.colorbar(cbar, ax=ax[0])
ax[1].imshow(confidence_region_slice.T, origin="lower", extent=[θ_0_min, θ_0_max, Θ_min[0], Θ_max[0]], aspect="auto", cmap="gray")
ax[1].set_xlabel(r"$\theta_0$")
ax[1].set_ylabel(r"$\theta_1$")
ax[1].set_title(r"$\Lambda \leq \chi_{3,0.95}^2$");

In [ ]:
# We don't plot the confidence bounds here because the figure would become too messy.
θ_0_confidence, Θ_confidence = θ_0_mesh[confidence_region], [θ[confidence_region] for θ in Θ_mesh]

## Figure 14

In [ ]:
As = np.linspace(1, 1500, 500)

fig, ax = plt.subplots(1, 1, figsize=(text_width, 0.6 * text_width))
ax.grid(True, alpha=0.3)

ax.plot(As, pod_models.PoD(np.stack((np.log(As), np.log(17.0) * np.ones_like(As)), axis=-1), θ_0_MLE_AR, Θ_MLE_AR), color=color_17_0, label="$r = 17.0$ pix/mm")
ax.plot(As, pod_models.PoD(np.stack((np.log(As), np.log(3.0) * np.ones_like(As)), axis=-1), θ_0_MLE_AR, Θ_MLE_AR), color=color_3_0, label="$r = 3.0$ pix/mm")
ax.plot(As, pod_models.PoD(np.stack((np.log(As), np.log(1.5) * np.ones_like(As)), axis=-1), θ_0_MLE_AR, Θ_MLE_AR), color=color_1_5, label="$r = 1.5$ pix/mm")
ax.plot(As, pod_models.PoD(np.stack((np.log(As), np.log(0.5) * np.ones_like(As)), axis=-1), θ_0_MLE_AR, Θ_MLE_AR), color=color_0_5, label="$r = 0.5$ pix/mm")
ax.plot(As, pod_models.DNVGL(As), color=color_DNVGL, linestyle=(5, (2, 3)), label="DNVGL")
ax.plot(As, pod_models.Campbell(As), color=color_Campbell, linestyle=(5, (2, 3)), label="Campbell")

ax.set_xticks(np.arange(0, 1501, 100))
ax.set_yticks(np.linspace(0, 1, 11))
ax.set_xlim([-50, 1500])
ax.tick_params(axis='x', rotation=45)
ax.set_xlabel("Crack Length (mm)")
ax.set_ylabel("Probability of Detection")
ax.legend(loc='lower right')
fig.savefig("Figures/PoD_resolution_parametric.png")

# Comparison Binning and Parametric Models

In [ ]:
def residual(x, θ_0, Θ, z):
    return ((pod_models.PoD(x, θ_0, Θ) - z)**2).sum()

residuals_parametric = residual(X_AR, θ_0_MLE_AR, Θ_MLE_AR, Z) / N
residuals_A_0_5 = residual(X_A_0_5, θ_0_MLE_A_0_5, Θ_MLE_A_0_5, Z_0_5)
residuals_A_1_5 = residual(X_A_1_5, θ_0_MLE_A_1_5, Θ_MLE_A_1_5, Z_1_5)
residuals_A_3_0 = residual(X_A_3_0, θ_0_MLE_A_3_0, Θ_MLE_A_3_0, Z_3_0)
residuals_A_17_0 = residual(X_A_17_0, θ_0_MLE_A_17_0, Θ_MLE_A_17_0, Z_17_0)
residuals_binned = (residuals_A_0_5 + residuals_A_1_5 + residuals_A_3_0 + residuals_A_17_0) / N

print(f"Residuals parametric: {residuals_parametric:.3f}, Residuals binning: {residuals_binned:.3f}")

## Figure 15

In [ ]:
As = np.linspace(1, 1500, 500)

fig, ax = plt.subplots(1, 1, figsize=(text_width, 0.6 * text_width))
ax.grid(True, alpha=0.3)

ax.plot(As, pod_models.PoD(np.stack((np.log(As), np.log(17.0) * np.ones_like(As)), axis=-1), θ_0_MLE_AR, Θ_MLE_AR), color=color_17_0, label="$r = 17.0$ pix/mm")
ax.plot(As, pod_models.PoD(np.stack((np.log(As), np.log(3.0) * np.ones_like(As)), axis=-1), θ_0_MLE_AR, Θ_MLE_AR), color=color_3_0, label="$r = 3.0$ pix/mm")
ax.plot(As, pod_models.PoD(np.stack((np.log(As), np.log(1.5) * np.ones_like(As)), axis=-1), θ_0_MLE_AR, Θ_MLE_AR), color=color_1_5, label="$r = 1.5$ pix/mm")
ax.plot(As, pod_models.PoD(np.stack((np.log(As), np.log(0.5) * np.ones_like(As)), axis=-1), θ_0_MLE_AR, Θ_MLE_AR), color=color_0_5, label="$r = 0.5$ pix/mm")
ax.plot(As, pod_models.PoD(np.log(As)[..., None], θ_0_MLE_A_17_0, Θ_MLE_A_17_0), color=color_17_0, linestyle=":", label="$r = 17.0$ pix/mm")
ax.plot(As, pod_models.PoD(np.log(As)[..., None], θ_0_MLE_A_3_0, Θ_MLE_A_3_0), color=color_3_0, linestyle=":", label="$r = 3.0$ pix/mm")
ax.plot(As, pod_models.PoD(np.log(As)[..., None], θ_0_MLE_A_1_5, Θ_MLE_A_1_5), color=color_1_5, linestyle=":", label="$r = 1.5$ pix/mm")
ax.plot(As, pod_models.PoD(np.log(As)[..., None], θ_0_MLE_A_0_5, Θ_MLE_A_0_5), color=color_0_5, linestyle=":", label="$r = 0.5$ pix/mm")

ax.set_xticks(np.arange(0, 1501, 100))
ax.set_yticks(np.linspace(0, 1, 11))
ax.set_xlim([-50, 1500])
ax.tick_params(axis='x', rotation=45)
ax.set_xlabel("Crack Length (mm)")
ax.set_ylabel("Probability of Detection")

style_handles = [
    Line2D([0], [0], color="black", lw=2, linestyle="-", label="Parametric"),
    Line2D([0], [0], color="black", lw=2, linestyle=":", label="Binned"),
]
leg1 = ax.legend(
    handles=style_handles,
    loc="lower right",
    bbox_to_anchor=(1.0, 0.4),
)
ax.add_artist(leg1)

color_handles = [
    Line2D([0], [0], color=color_17_0, lw=2, label=r"$r = 17.0$ pix/mm"),
    Line2D([0], [0], color=color_3_0,  lw=2, label=r"$r = 3.0$ pix/mm"),
    Line2D([0], [0], color=color_1_5,  lw=2, label=r"$r = 1.5$ pix/mm"),
    Line2D([0], [0], color=color_0_5,  lw=2, label=r"$r = 0.5$ pix/mm"),
]
ax.legend(
    handles=color_handles,
    loc="lower right",
    bbox_to_anchor=(1.0, 0.0),
)

fig.savefig("Figures/PoD_parametric_vs_binning.png")